# SmartVision AI — retrain all (classification + YOLO + comparison)

**One Colab notebook. One T4 runtime. Runtime → Run all.**

Do **not** rebuild the dataset. Reuses:
- `MyDrive/smartvision_dataset.zip`
- `MyDrive/SmartVision_artifacts/models/*.keras` and `yolov8_best.pt` (run-1 is copied to `*_run1` first)

TensorFlow is capped at **8 GB** so YOLO can use the rest of the T4 after the CNNs. Stay on this tab — Colab still kills idle sessions, but you should **not** need to change notebooks or factory-reset between phases.

If it disconnects after CNNs finish: reconnect, set `SKIP_CLASSIFICATION = True` (skips only the four train cells). Still run setup + tf.data + helpers + recovery + everything after.


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install tensorflow pandas scikit-learn matplotlib seaborn pillow tqdm ultralytics pyyaml opencv-python-headless
print("deps ready")


In [ ]:
# If Colab died after a section, reconnect, set that flag True, Run all.
SKIP_CLASSIFICATION = False  # True = CNN keras + classification_metrics.json already updated on Drive
SKIP_YOLO = False            # True = yolo_metrics.json already updated on Drive
SKIP_COMPARISON = False


In [ ]:
import os, sys, json, time, random, shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, applications, optimizers, callbacks

print("TF", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    # Cap TF at 8GB so YOLOv8 (PyTorch) can use the rest of the T4 in this same runtime
    try:
        tf.config.set_logical_device_configuration(
            gpus[0],
            [tf.config.LogicalDeviceConfiguration(memory_limit=8192)],
        )
        print("TF GPU memory limit = 8192 MB (leave headroom for YOLO)")
    except Exception as e:
        print("Could not cap TF GPU (already initialized?):", e)
        try:
            tf.config.experimental.set_memory_growth(gpus[0], True)
        except Exception as e2:
            print("memory_growth also skipped:", e2)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    if zip_path.exists() and not (data_dir / "classification" / "train").exists():
        import zipfile
        print("Unzipping dataset (reuse Drive zip, not a new collect)...")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_dir if not any(data_dir.glob("*")) else PROJECT_ROOT)
        if not (data_dir / "classification").exists():
            inner = PROJECT_ROOT / "smartvision_dataset"
            print("classification path exists:", (inner / "classification").exists())
    DRIVE_REPO = Path("/content/drive/MyDrive/Smart_Vision_AI")
    if DRIVE_REPO.exists():
        sys.path.insert(0, str(DRIVE_REPO))
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    sys.path.insert(0, str(PROJECT_ROOT))
    OUT_ROOT = PROJECT_ROOT

CLASS_NAMES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "truck",
    "traffic light", "stop sign", "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl", "pizza", "cake", "chair",
    "couch", "potted plant", "bed",
]
NUM_CLASSES = 25
IMAGE_SIZE = 224
BATCH = 16  # 16 is safer on Colab T4 with VGG16 flatten; raise to 32 if memory allows

DATA = PROJECT_ROOT / "smartvision_dataset" / "classification"
for alt in (
    Path("/content/smartvision_dataset/classification"),
    Path("/content/drive/MyDrive/smartvision_dataset/classification"),
    Path("/content/drive/MyDrive/SmartVision_artifacts/smartvision_dataset/classification"),
):
    if (DATA / "train").exists():
        break
    if (alt / "train").exists():
        DATA = alt
        break
print("DATA =", DATA, "exists", DATA.exists())

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

print("OUT_ROOT =", OUT_ROOT)
print("Existing checkpoints (resume stage 2 from these if FOUND):")
for stem in ("vgg16", "resnet50", "mobilenetv2", "efficientnetb0"):
    p = MODELS_DIR / f"{stem}.keras"
    if p.exists():
        print(f"  {p.name}: FOUND {p.stat().st_size/1e6:.1f} MB")
    else:
        print(f"  {p.name}: missing — will train stage 1 from ImageNet")

def count_split(split):
    rows = {}
    root = DATA / split
    for n in CLASS_NAMES:
        folder = root / n
        rows[n] = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    return rows

for split in ("train", "val", "test"):
    c = count_split(split)
    print(f"{split:5s} total={sum(c.values()):4d}  min_class={min(c.values())} max_class={max(c.values())} missing={[k for k,v in c.items() if v==0]}")
assert sum(count_split("train").values()) > 0, "No training images. Unzip MyDrive/smartvision_dataset.zip — do not rebuild notebook 1."


In [ ]:
## tf.data pipelines + the brief's augmentation set
# Always run this cell (needed for recovery eval even if CNN training is skipped).

def list_image_label(split):
    paths, labels = [], []
    root = DATA / split
    for idx, name in enumerate(CLASS_NAMES):
        for f in sorted((root / name).glob("*.jpg")):
            paths.append(str(f))
            labels.append(idx)
    return tf.constant(paths), tf.constant(labels, dtype=tf.int32)

def decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15.0 / 360.0, fill_mode="nearest"),
    layers.RandomBrightness(0.20),
    layers.RandomContrast(0.20),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.05, 0.05),
], name="brief_augmentation")

def color_jitter(img, label):
    img = tf.image.random_saturation(img / 255.0, 0.7, 1.3) * 255.0
    img = tf.image.random_hue(img / 255.0, 0.05) * 255.0
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_ds(split, training=False, batch=BATCH, extra_advanced=False):
    paths, labels = list_image_label(split)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.map(color_jitter, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds, len(paths)

train_ds, n_train = make_ds("train", training=True)
val_ds, n_val = make_ds("val", training=False)
test_ds, n_test = make_ds("test", training=False)
print(f"n_train={n_train} n_val={n_val} n_test={n_test}")
print("test_ds ready")


In [ ]:
def mixup_ds(ds, alpha=0.2):
    # Advanced MixUp for EfficientNet stage 1 only (bonus +2) if training from scratch
    def _mix(batch_x, batch_y):
        g1 = tf.random.gamma([], alpha)
        g2 = tf.random.gamma([], alpha)
        lam = g1 / (g1 + g2)
        idx = tf.random.shuffle(tf.range(tf.shape(batch_x)[0]))
        mixed_x = lam * batch_x + (1.0 - lam) * tf.gather(batch_x, idx)
        y1 = tf.one_hot(batch_y, NUM_CLASSES)
        y2 = tf.one_hot(tf.gather(batch_y, idx), NUM_CLASSES)
        mixed_y = lam * y1 + (1.0 - lam) * y2
        return mixed_x, mixed_y
    return ds.map(_mix, num_parallel_calls=tf.data.AUTOTUNE)


def clf_loss():
    try:
        return keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1)
    except TypeError:
        return "sparse_categorical_crossentropy"


def common_callbacks(name, patience=8, csv_name=None, min_val_acc=None):
    csv = csv_name or name
    ckpt_kw = dict(
        filepath=str(MODELS_DIR / f"{name}.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1,
    )
    try:
        if min_val_acc is not None:
            ckpt = callbacks.ModelCheckpoint(initial_value_threshold=float(min_val_acc), **ckpt_kw)
        else:
            ckpt = callbacks.ModelCheckpoint(**ckpt_kw)
    except TypeError:
        ckpt = callbacks.ModelCheckpoint(**ckpt_kw)
    return [
        callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True, mode="max"),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        ckpt,
        callbacks.CSVLogger(str(REPORTS / f"{csv}_history.csv")),
    ]


def plot_history(hist, name):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.history["accuracy"], label="train")
    ax[0].plot(hist.history["val_accuracy"], label="val")
    ax[0].set_title(f"{name} accuracy"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(hist.history["loss"], label="train")
    ax[1].plot(hist.history["val_loss"], label="val")
    ax[1].set_title(f"{name} loss"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES / f"{name}_history.png", dpi=130)
    plt.show()


def find_backbone(model, substr):
    key = substr.lower()
    for layer in model.layers:
        if key in layer.name.lower():
            return layer
    raise RuntimeError(f"No backbone matching {substr!r} in {[l.name for l in model.layers]}")


def unfreeze_last(backbone, n_unfreeze, freeze_bn=True):
    backbone.trainable = True
    keep = max(0, len(backbone.layers) - n_unfreeze)
    for i, layer in enumerate(backbone.layers):
        layer.trainable = i >= keep
        if freeze_bn and isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    n_tr = sum(1 for l in backbone.layers if l.trainable)
    print(f"backbone {backbone.name}: trainable {n_tr}/{len(backbone.layers)} (last {n_unfreeze} opened, BN frozen={freeze_bn})")


def compile_clf(model, lr):
    model.compile(
        optimizer=optimizers.Adam(lr),
        loss=clf_loss(),
        metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")],
    )


def backup_run1(stem):
    src = MODELS_DIR / f"{stem}.keras"
    dst = MODELS_DIR / f"{stem}_run1.keras"
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print("Backed up run-1 weights ->", dst)


def val_accuracy(model):
    out = model.evaluate(val_ds, verbose=0)
    # [loss, accuracy, top5] or [loss, accuracy]
    acc = float(out[1])
    print(f"current val_accuracy={acc:.4f}")
    return acc


def load_or_none(stem):
    path = MODELS_DIR / f"{stem}.keras"
    if path.exists():
        print("Loading Drive checkpoint", path)
        keras.mixed_precision.set_global_policy("float32")
        return keras.models.load_model(path)
    return None


def stage2_finetune(model, stem, backbone_substr, n_unfreeze, epochs, lr, freeze_bn=True):
    backup_run1(stem)
    bb = find_backbone(model, backbone_substr)
    unfreeze_last(bb, n_unfreeze, freeze_bn=freeze_bn)
    compile_clf(model, lr)
    floor = val_accuracy(model)
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=common_callbacks(stem, patience=8, csv_name=f"{stem}_stage2", min_val_acc=floor),
        verbose=1,
    )
    plot_history(hist, f"{stem}_stage2")
    return keras.models.load_model(MODELS_DIR / f"{stem}.keras")


def evaluate_model(model, name):
    y_true, y_prob = [], []
    t0 = time.perf_counter()
    n = 0
    for xb, yb in test_ds:
        p = model.predict(xb, verbose=0)
        y_true.append(yb.numpy())
        y_prob.append(p)
        n += xb.shape[0]
    elapsed = time.perf_counter() - t0
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    y_pred = y_prob.argmax(axis=1)
    top5 = np.mean([yt in np.argsort(pr)[-5:] for yt, pr in zip(y_true, y_prob)])
    acc = float(accuracy_score(y_true, y_pred))
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    prec_c, rec_c, f1_c, sup = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(11, 10))
    sns.heatmap(cm, cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"{name} confusion matrix (test)")
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    fig.tight_layout()
    fig.savefig(FIGURES / f"cm_{name}.png", dpi=140)
    plt.show()
    size_mb = (MODELS_DIR / f"{name}.keras").stat().st_size / 1e6 if (MODELS_DIR / f"{name}.keras").exists() else 0.0
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)
    print(report)
    per_class = {
        CLASS_NAMES[i]: {"precision": float(prec_c[i]), "recall": float(rec_c[i]), "f1": float(f1_c[i]), "support": int(sup[i])}
        for i in range(NUM_CLASSES)
    }
    metrics = {
        "model": name,
        "accuracy": acc,
        "precision_macro": float(prec),
        "recall_macro": float(rec),
        "f1_macro": float(f1),
        "top5_accuracy": float(top5),
        "inference_ms": float(elapsed / max(n, 1) * 1000),
        "model_size_mb": float(size_mb),
        "n_test": int(n),
        "per_class": per_class,
    }
    print(name, {k: metrics[k] for k in ("accuracy", "f1_macro", "top5_accuracy", "inference_ms", "model_size_mb")})
    return metrics, cm

ALL_METRICS = {}
print("Helpers ready. Label smoothing = 0.1; stage 2 will resume from Drive *.keras when present.")


## Phase 2 — four CNNs (resume Drive checkpoints, then stage-2 unfreeze)


In [ ]:
if SKIP_CLASSIFICATION:
    print('SKIP c_vgg')
else:
    ## Model 1 — VGG16: resume Drive checkpoint if present, else frozen head, then unfreeze last 8 (block5)

    keras.mixed_precision.set_global_policy("float32")
    vgg = load_or_none("vgg16")
    if vgg is None:
        print("No checkpoint — stage 1: frozen VGG16 conv base + dropout head")
        base = applications.VGG16(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
        base.trainable = False
        inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
        x = applications.vgg16.preprocess_input(inp)
        x = base(x, training=False)
        x = layers.Flatten()(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.Dropout(0.5)(x)
        out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        vgg = models.Model(inp, out, name="VGG16")
        compile_clf(vgg, 1e-3)
        hist_vgg1 = vgg.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("vgg16", patience=5), verbose=1)
        plot_history(hist_vgg1, "vgg16")
        vgg = keras.models.load_model(MODELS_DIR / "vgg16.keras")
    else:
        print("Skipping stage 1 — using run-1 VGG16 weights from Drive")

    print("Stage 2: unfreeze last 8 VGG layers, lr=1e-5")
    vgg = stage2_finetune(vgg, "vgg16", "vgg16", n_unfreeze=8, epochs=25, lr=1e-5, freeze_bn=False)
    ALL_METRICS["VGG16"], _ = evaluate_model(vgg, "vgg16")


In [ ]:
if SKIP_CLASSIFICATION:
    print('SKIP c_resnet')
else:
    ## Model 2 — ResNet50: resume Drive checkpoint if present, else last-20 warmup, then unfreeze last 40

    keras.mixed_precision.set_global_policy("float32")
    resnet = load_or_none("resnet50")
    if resnet is None:
        print("No checkpoint — stage 1: unfreeze last 20 (brief default) + GAP head")
        base = applications.ResNet50(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
        for layer in base.layers:
            layer.trainable = False
        for layer in base.layers[-20:]:
            layer.trainable = True
        inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
        x = applications.resnet50.preprocess_input(inp)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.Dropout(0.4)(x)
        out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        resnet = models.Model(inp, out, name="ResNet50")
        compile_clf(resnet, 1e-4)
        hist_rn1 = resnet.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("resnet50", patience=5), verbose=1)
        plot_history(hist_rn1, "resnet50")
        resnet = keras.models.load_model(MODELS_DIR / "resnet50.keras")
    else:
        print("Skipping stage 1 — using run-1 ResNet50 weights from Drive (was 76.5% test)")

    print("Stage 2: unfreeze last 40 ResNet layers, lr=1e-5, BN frozen")
    resnet = stage2_finetune(resnet, "resnet50", "resnet50", n_unfreeze=40, epochs=30, lr=1e-5, freeze_bn=True)
    ALL_METRICS["ResNet50"], _ = evaluate_model(resnet, "resnet50")


In [ ]:
if SKIP_CLASSIFICATION:
    print('SKIP c_mnet')
else:
    ## Model 3 — MobileNetV2: resume Drive checkpoint if present, else frozen head, then unfreeze last 40

    keras.mixed_precision.set_global_policy("float32")
    mnet = load_or_none("mobilenetv2")
    if mnet is None:
        print("No checkpoint — stage 1: frozen MobileNetV2 + compact head")
        base = applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
        base.trainable = False
        inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
        x = applications.mobilenet_v2.preprocess_input(inp)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(128, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
        out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
        mnet = models.Model(inp, out, name="MobileNetV2")
        compile_clf(mnet, 1e-3)
        hist_mn1 = mnet.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=common_callbacks("mobilenetv2", patience=5), verbose=1)
        plot_history(hist_mn1, "mobilenetv2")
        mnet = keras.models.load_model(MODELS_DIR / "mobilenetv2.keras")
    else:
        print("Skipping stage 1 — using run-1 MobileNetV2 weights from Drive")

    print("Stage 2: unfreeze last 40 MobileNet layers, lr=1e-5, BN frozen")
    mnet = stage2_finetune(mnet, "mobilenetv2", "mobilenet", n_unfreeze=40, epochs=25, lr=1e-5, freeze_bn=True)
    ALL_METRICS["MobileNetV2"], _ = evaluate_model(mnet, "mobilenetv2")


In [ ]:
if SKIP_CLASSIFICATION:
    print('SKIP c_eff')
else:
    ## Model 4 — EfficientNetB0: resume Drive checkpoint if present, else MixUp frozen + unfreeze last 50

    keras.mixed_precision.set_global_policy("float32")
    eff = load_or_none("efficientnetb0")
    if eff is None:
        print("No checkpoint — stage 1: frozen EfficientNet + BN head + MixUp")
        keras.mixed_precision.set_global_policy("mixed_float16")
        base = applications.EfficientNetB0(weights="imagenet", include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
        base.trainable = False
        inp = layers.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
        x = applications.efficientnet.preprocess_input(inp)
        x = base(x, training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dense(256, activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.4)(x)
        out = layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
        eff = models.Model(inp, out, name="EfficientNetB0")
        train_mix = mixup_ds(train_ds)
        eff.compile(optimizer=optimizers.Adam(1e-4), loss="categorical_crossentropy", metrics=["accuracy"])
        hist_eff1 = eff.fit(
            train_mix,
            validation_data=val_ds.map(lambda x, y: (x, tf.one_hot(y, NUM_CLASSES))),
            epochs=10,
            callbacks=common_callbacks("efficientnetb0", patience=5),
            verbose=1,
        )
        plot_history(hist_eff1, "efficientnetb0")
        keras.mixed_precision.set_global_policy("float32")
        eff = keras.models.load_model(MODELS_DIR / "efficientnetb0.keras")
    else:
        print("Skipping stage 1 — using run-1 EfficientNetB0 weights from Drive")

    print("Stage 2: unfreeze last 50 EfficientNet layers, lr=1e-5, BN frozen")
    eff = stage2_finetune(eff, "efficientnetb0", "efficientnet", n_unfreeze=50, epochs=25, lr=1e-5, freeze_bn=True)
    ALL_METRICS["EfficientNetB0"], _ = evaluate_model(eff, "efficientnetb0")


In [ ]:
# Recovery: load scores from Drive keras files, then unfreeze any model still under 80%.
# Safe after a kernel reconnect. Do NOT rerun the four CNN training cells.

if "ALL_METRICS" not in dir() or not ALL_METRICS:
    ALL_METRICS = {}

for display_name, stem in (
    ("VGG16", "vgg16"),
    ("ResNet50", "resnet50"),
    ("MobileNetV2", "mobilenetv2"),
    ("EfficientNetB0", "efficientnetb0"),
):
    if display_name not in ALL_METRICS and (MODELS_DIR / f"{stem}.keras").exists():
        print("Evaluating", stem, "from Drive")
        ALL_METRICS[display_name], _ = evaluate_model(
            keras.models.load_model(MODELS_DIR / f"{stem}.keras"), stem
        )

def deepen_finetune(stem, substr, n_unfreeze, lr, epochs=15, freeze_bn=True):
    model = keras.models.load_model(MODELS_DIR / f"{stem}.keras")
    bb = find_backbone(model, substr)
    unfreeze_last(bb, n_unfreeze, freeze_bn=freeze_bn)
    compile_clf(model, lr)
    floor = val_accuracy(model)
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=common_callbacks(stem, patience=6, csv_name=f"{stem}_recovery", min_val_acc=floor),
        verbose=1,
    )
    return keras.models.load_model(MODELS_DIR / f"{stem}.keras")

RECOVERY = {
    "VGG16": ("vgg16", "vgg16", 16, 5e-6, False),
    "ResNet50": ("resnet50", "resnet50", 80, 5e-6, True),
    "MobileNetV2": ("mobilenetv2", "mobilenet", 80, 5e-6, True),
    "EfficientNetB0": ("efficientnetb0", "efficientnet", 120, 3e-6, True),
}
for display_name, (file_stem, substr, n_unfreeze, lr, freeze_bn) in RECOVERY.items():
    acc = ALL_METRICS[display_name]["accuracy"]
    print(f"{display_name} test acc={acc:.3f}")
    if acc < 0.80:
        print(f"  -> below 80%. Unfreezing last {n_unfreeze} backbone layers (lr={lr}).")
        m = deepen_finetune(file_stem, substr, n_unfreeze, lr, epochs=15, freeze_bn=freeze_bn)
        ALL_METRICS[display_name], _ = evaluate_model(m, file_stem)
    else:
        print("  -> meets 80% floor; leave weights as-is.")


In [ ]:
## Persist classification metrics from THIS session's ALL_METRICS (the Drive keras eval).
# Do not reload the old JSON if we just scored the new weights.

if not ALL_METRICS or len(ALL_METRICS) < 4:
    payload = json.loads((REPORTS / "classification_metrics.json").read_text(encoding="utf-8"))
    ALL_METRICS = payload["classification"]
    n_train = payload.get("n_train", n_train if "n_train" in dir() else 0)
    n_val = payload.get("n_val", n_val if "n_val" in dir() else 0)
    n_test = payload.get("n_test", n_test if "n_test" in dir() else 0)
    print("ALL_METRICS was empty — loaded classification_metrics.json from Drive")

payload = {
    "class_names": CLASS_NAMES,
    "n_train": int(n_train),
    "n_val": int(n_val),
    "n_test": int(n_test),
    "classification": ALL_METRICS,
    "best_classification_model": max(ALL_METRICS, key=lambda k: ALL_METRICS[k]["accuracy"]),
}
(REPORTS / "classification_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != "per_class"} for k, v in ALL_METRICS.items()}, indent=2))
print("Best:", payload["best_classification_model"])
print("Saved models:", sorted(p.name for p in MODELS_DIR.glob("*.keras")))
print("Paste the printed accuracy table back into chat when this cell finishes.")


## Free TensorFlow GPU memory before YOLO (same runtime, no restart)


In [ ]:
import gc

for name in ("vgg", "resnet", "mnet", "eff", "train_ds", "val_ds", "test_ds", "xb", "yb"):
    if name in globals():
        del globals()[name]

keras.backend.clear_session()
gc.collect()

try:
    import torch
    torch.cuda.empty_cache()
    print("torch cuda allocated MB", round(torch.cuda.memory_allocated() / 1e6, 1))
    print("torch cuda reserved MB", round(torch.cuda.memory_reserved() / 1e6, 1))
except Exception as e:
    print("torch cache:", e)

print("TF GPU devices", tf.config.list_logical_devices("GPU"))
print("Ready for YOLO in this same runtime.")


## Phase 3 — YOLOv8m freeze-10 fine-tune (same 25-class Drive zip)


In [ ]:
import os, sys, json, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/Smart_Vision_AI")
    PROJECT_ROOT.mkdir(exist_ok=True)
    zip_path = Path("/content/drive/MyDrive/smartvision_dataset.zip")
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
else:
    PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    zip_path = PROJECT_ROOT / "smartvision_dataset.zip"
    data_dir = PROJECT_ROOT / "smartvision_dataset"
    OUT_ROOT = PROJECT_ROOT

print("zip exists:", zip_path.exists(), zip_path)
if zip_path.exists():
    print("zip size MB:", round(zip_path.stat().st_size / 1e6, 1))


def unzip_smartvision(zip_path, project_root, data_dir):
    """Notebook 1 zips the *contents* of smartvision_dataset, so the archive
    has classification/ and detection/ at the root — extract into data_dir."""
    with zipfile.ZipFile(zip_path) as z:
        names = [n.replace("\\", "/") for n in z.namelist()]
        print("zip sample:", names[:12])
        nested = any(n.startswith("smartvision_dataset/") for n in names)
        dest = project_root if nested else data_dir
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting to", dest)
        z.extractall(dest)


def find_detection_dir(project_root):
    candidates = [
        project_root / "smartvision_dataset" / "detection",
        project_root / "detection",
        Path("/content/smartvision_dataset/detection"),
        Path("/content/detection"),
        Path("/content/drive/MyDrive/smartvision_dataset/detection"),
        Path("/content/drive/MyDrive/SmartVision_artifacts/smartvision_dataset/detection"),
    ]
    print("Looking for data.yaml in:")
    for c in candidates:
        ok = (c / "data.yaml").exists()
        print(" ", "FOUND" if ok else "miss ", c)
        if ok:
            return c
    for root in (project_root, Path("/content"), Path("/content/drive/MyDrive")):
        if not root.exists():
            continue
        hits = list(root.rglob("data.yaml"))[:20]
        print("rglob data.yaml under", root, "->", hits)
        for h in hits:
            if h.parent.name == "detection":
                return h.parent
    return None


if zip_path.exists() and not (data_dir / "detection" / "data.yaml").exists():
    print("Unzipping (reuse Drive zip)...")
    unzip_smartvision(zip_path, PROJECT_ROOT, data_dir)
elif not zip_path.exists():
    print("WARNING: Drive zip not found. Finish Smartvision.ipynb last cell first.")

DET = find_detection_dir(PROJECT_ROOT)
print("DET =", DET)

MODELS_DIR = OUT_ROOT / "models"
FIGURES = OUT_ROOT / "reports" / "figures"
REPORTS = OUT_ROOT / "reports"
for p in (MODELS_DIR, FIGURES, REPORTS):
    p.mkdir(parents=True, exist_ok=True)

old_pt = MODELS_DIR / "yolov8_best.pt"
run1_pt = MODELS_DIR / "yolov8_best_run1.pt"
if old_pt.exists() and not run1_pt.exists():
    shutil.copy2(old_pt, run1_pt)
    print("Backed up run-1 YOLO ->", run1_pt)
print("Existing YOLO files:")
for p in sorted(MODELS_DIR.glob("yolo*")):
    print(" ", p.name, f"{p.stat().st_size/1e6:.1f} MB")

if DET is None or not (DET / "data.yaml").exists():
    raise FileNotFoundError(
        "data.yaml not found. Unzip MyDrive/smartvision_dataset.zip — do not rebuild notebook 1."
    )

yaml_text = (DET / "data.yaml").read_text(encoding="utf-8")
print(yaml_text)

import yaml
cfg = yaml.safe_load(yaml_text)
cfg["path"] = str(DET.resolve())
assert int(cfg.get("nc", 0)) == 25, cfg
names = cfg["names"]
if isinstance(names, dict):
    class_names = [names[i] for i in range(len(names))]
else:
    class_names = list(names)
assert "train" not in class_names, class_names
assert len(class_names) == 25
(DET / "data.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
print("Updated data.yaml path ->", cfg["path"])
print("names:", class_names)

RUN_NAME = "smartvision_yolov8m_freeze"
print("New Ultralytics run folder:", OUT_ROOT / "yolo_runs" / RUN_NAME)
print("Old run-1 folder is left untouched: yolo_runs/smartvision_yolov8s/")


In [ ]:
## Verify YOLO labels against images (data-driven sanity, not assumed)

import yaml
from pathlib import Path
from collections import Counter

# Recover DET / class_names if the previous cell was the short unzip snippet
if "DET" not in dir() or DET is None:
    hits = list(Path("/content").rglob("data.yaml"))
    DET = next((h.parent for h in hits if h.parent.name == "detection"), None)
    print("Recovered DET =", DET)
assert DET is not None and (DET / "data.yaml").exists(), DET

if "class_names" not in dir():
    cfg = yaml.safe_load((DET / "data.yaml").read_text(encoding="utf-8"))
    names = cfg["names"]
    class_names = [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)
    print("Loaded class_names from yaml:", class_names)

print("DET =", DET)
print("n classes =", len(class_names))

def verify_split(split):
    img_dir = DET / "images" / split
    lab_dir = DET / "labels" / split
    images = sorted(img_dir.glob("*.jpg"))
    missing, bad, objs = [], 0, 0
    per_class = Counter()
    for imgp in images:
        lab = lab_dir / (imgp.stem + ".txt")
        if not lab.exists():
            missing.append(imgp.name)
            continue
        for line in lab.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 5:
                bad += 1
                continue
            cls, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
            if not (0 <= cls < 25):
                bad += 1
                continue
            if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1):
                bad += 1
                continue
            objs += 1
            per_class[class_names[cls]] += 1
    return {"n_images": len(images), "n_labels": len(list(lab_dir.glob("*.txt"))),
            "missing_labels": len(missing), "bad_rows": bad, "objects": objs, "per_class": dict(per_class)}

for split in ("train", "val", "test"):
    stats = verify_split(split)
    print(split, {k: stats[k] for k in stats if k != "per_class"})
print("\\nTrain objects per class:")
print(pd.Series(verify_split("train")["per_class"]).reindex(class_names))

In [ ]:
if SKIP_YOLO:
    print('SKIP c_yolo_train')
else:
    print('If CUDA OOM: change BATCH to 4 in this cell and re-run it.')
    ## Train YOLOv8m with frozen backbone (keep COCO features; new 25-class head)

    from ultralytics import YOLO

    # If T4 OOMs, drop batch from 8 to 4 and re-run this cell only.
    BATCH = 4  # 4 in this combined notebook (TF still holds 8GB)
    RUN_NAME = "smartvision_yolov8m_freeze"

    model = YOLO("yolov8m.pt")
    results = model.train(
        data=str(DET / "data.yaml"),
        epochs=40,
        imgsz=640,
        batch=BATCH,
        freeze=10,
        patience=15,
        seed=42,
        workers=2,
        project=str(OUT_ROOT / "yolo_runs"),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer="AdamW",
        lr0=0.0003,
        lrf=0.01,
        close_mosaic=10,
        mixup=0.1,
        plots=True,
        verbose=True,
    )
    print(results)
    print("Train finished. Next cell evaluates val mAP@0.5 (need > 0.75).")


In [ ]:
if SKIP_YOLO:
    print('SKIP c_yolo_eval')
else:
    ## Evaluate on val (primary / rubric) and test (held-out report)

    RUN_NAME = "smartvision_yolov8m_freeze"
    best = Path(OUT_ROOT / "yolo_runs" / RUN_NAME / "weights" / "best.pt")
    print("best exists", best.exists(), best)
    if not best.exists():
        raise FileNotFoundError(f"Missing {best}. Re-run the train cell; check yolo_runs/{RUN_NAME}/weights/")

    from ultralytics import YOLO
    trained = YOLO(str(best))

    val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
    print("VAL", val_metrics.results_dict if hasattr(val_metrics, "results_dict") else val_metrics)

    test_metrics = trained.val(data=str(DET / "data.yaml"), split="test", plots=True)
    print("TEST", test_metrics.results_dict if hasattr(test_metrics, "results_dict") else test_metrics)

    def extract(m):
        d = m.results_dict if hasattr(m, "results_dict") else {}
        return {
            "map50": float(d.get("metrics/mAP50(B)", getattr(getattr(m, "box", None), "map50", 0.0) or 0.0)),
            "map50_95": float(d.get("metrics/mAP50-95(B)", getattr(getattr(m, "box", None), "map", 0.0) or 0.0)),
            "precision": float(d.get("metrics/precision(B)", getattr(getattr(m, "box", None), "mp", 0.0) or 0.0)),
            "recall": float(d.get("metrics/recall(B)", getattr(getattr(m, "box", None), "mr", 0.0) or 0.0)),
            "raw": {k: float(v) if isinstance(v, (int, float, np.floating)) else str(v) for k, v in d.items()},
        }

    val_ex = extract(val_metrics)
    test_ex = extract(test_metrics)
    print("VAL extracted", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
    print("TEST extracted", {k: test_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
    print("RUBRIC: val mAP@0.5 > 0.75 ?", val_ex["map50"] > 0.75)

    ap_per_class = {}
    box = getattr(val_metrics, "box", None)
    if box is not None and hasattr(box, "ap_class_index") and hasattr(box, "ap50"):
        for idx, ap in zip(box.ap_class_index, box.ap50):
            ap_per_class[class_names[int(idx)]] = float(ap)
    print("per-class AP50", ap_per_class)


In [ ]:
if SKIP_YOLO:
    print('SKIP c_yolo_fps')
else:
    ## Speed (FPS) on a handful of val images

    import time
    val_imgs = sorted((DET / "images" / "val").glob("*.jpg"))[:50]
    # warmup
    _ = trained.predict(source=str(val_imgs[0]), verbose=False)
    t0 = time.perf_counter()
    _ = trained.predict(source=[str(p) for p in val_imgs], verbose=False)
    dt = time.perf_counter() - t0
    fps = len(val_imgs) / max(dt, 1e-6)
    print(f"{len(val_imgs)} images in {dt:.2f}s -> {fps:.1f} FPS")


In [ ]:
if SKIP_YOLO:
    print('SKIP c_yolo_viz')
else:
    ## Visualize predictions on sample val images

    samples = val_imgs[:8]
    pred = trained.predict(source=[str(p) for p in samples], conf=0.5, verbose=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, r, p in zip(axes.ravel(), pred, samples):
        plotted = r.plot()  # BGR ndarray
        ax.imshow(plotted[:, :, ::-1])
        ax.set_title(p.name, fontsize=8)
        ax.axis("off")
    fig.suptitle("YOLOv8 val predictions (conf>=0.5)")
    fig.tight_layout()
    fig.savefig(FIGURES / "yolo_val_samples.png", dpi=140)
    plt.show()


In [ ]:
if SKIP_YOLO:
    print("SKIP c_yolo_fail")
else:
    ## Failure gallery: one image at a time (avoids CUDA OOM on T4)

    import torch
    torch.cuda.empty_cache()

    fail, ok = [], []
    all_val = sorted((DET / "images" / "val").glob("*.jpg"))
    chunk = all_val[:40]
    for p in chunk:
        r = trained.predict(source=str(p), conf=0.5, verbose=False)[0]
        n = 0 if r.boxes is None else len(r.boxes)
        confs = [] if n == 0 else [float(c) for c in r.boxes.conf]
        mean_c = float(np.mean(confs)) if confs else 0.0
        rec = {"path": p, "n": n, "mean_conf": mean_c}
        (fail if n == 0 else ok).append(rec)
        del r
    torch.cuda.empty_cache()

    print(f"scanned {len(chunk)} val images; zero-detection={len(fail)}")
    ok.sort(key=lambda d: d["mean_conf"])
    gallery = fail[:8] + ok[: max(0, 8 - len(fail[:8]))]
    if gallery:
        fig, axes = plt.subplots(2, 4, figsize=(14, 7))
        for ax, rec in zip(axes.ravel(), gallery):
            r = trained.predict(source=str(rec["path"]), conf=0.25, verbose=False)[0]
            ax.imshow(r.plot()[:, :, ::-1])
            ax.set_title(f"n={rec['n']} conf={rec['mean_conf']:.2f}", fontsize=8)
            ax.axis("off")
            del r
        fig.suptitle("Failure / low-confidence cases (drawn at conf=0.25)")
        fig.tight_layout()
        fig.savefig(FIGURES / "yolo_failures.png", dpi=140)
        plt.show()
        torch.cuda.empty_cache()
    else:
        print("No failures in the scanned slice.")


In [ ]:
if SKIP_YOLO:
    print('SKIP c_yolo_save')
else:
    ## Copy best weights + write yolo_metrics.json
    # If val mAP50 is still <= 0.75, short unfrozen pass at low LR (does not delete the freeze run).

    map50 = val_ex["map50"]
    print("val mAP50 =", map50)
    if map50 <= 0.75:
        print("Below 75% floor — 15 more epochs from best.pt with freeze=0, lr0=1e-4")
        from ultralytics import YOLO
        model2 = YOLO(str(best))
        model2.train(
            data=str(DET / "data.yaml"),
            epochs=15,
            imgsz=640,
            batch=4,
            freeze=0,
            patience=8,
            seed=42,
            workers=2,
            project=str(OUT_ROOT / "yolo_runs"),
            name="smartvision_yolov8m_unfreeze",
            exist_ok=True,
            optimizer="AdamW",
            lr0=0.0001,
            lrf=0.01,
            plots=True,
        )
        extra = Path(OUT_ROOT / "yolo_runs" / "smartvision_yolov8m_unfreeze" / "weights" / "best.pt")
        if extra.exists():
            best = extra
            trained = YOLO(str(best))
            val_metrics = trained.val(data=str(DET / "data.yaml"), split="val", plots=True)
            val_ex = extract(val_metrics)
            test_metrics = trained.val(data=str(DET / "data.yaml"), split="test", plots=True)
            test_ex = extract(test_metrics)
            print("VAL after unfreeze pass", {k: val_ex[k] for k in ("map50", "map50_95", "precision", "recall")})
            print("RUBRIC after unfreeze: val mAP@0.5 > 0.75 ?", val_ex["map50"] > 0.75)

    dest = MODELS_DIR / "yolov8_best.pt"
    shutil.copy2(best, dest)
    print("Copied", dest)
    print("Run-1 backup still at", MODELS_DIR / "yolov8_best_run1.pt")

    yolo_payload = {
        "model": "YOLOv8m",
        "weights": str(dest),
        "val": {k: v for k, v in val_ex.items() if k != "raw"},
        "test": {k: v for k, v in test_ex.items() if k != "raw"},
        "fps": float(fps),
        "per_class_ap50": ap_per_class,
        "n_classes": 25,
        "class_names": class_names,
        "meets_map50_floor": bool(val_ex["map50"] > 0.75),
    }
    (REPORTS / "yolo_metrics.json").write_text(json.dumps(yolo_payload, indent=2), encoding="utf-8")
    print(json.dumps(yolo_payload, indent=2)[:2000])
    print("Paste VAL extracted mAP50 back into chat when done.")


## Phase 4 — comparison, pipeline, TFLite, metrics.json


In [ ]:
if SKIP_COMPARISON:
    print('SKIP c_cmp_load')
else:
    import os, sys, json, time
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from PIL import Image, ImageDraw, ImageFont

    IN_COLAB = "google.colab" in sys.modules
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        PROJECT_ROOT = Path("/content/Smart_Vision_AI")
        OUT_ROOT = Path("/content/drive/MyDrive/SmartVision_artifacts")
        sys.path.insert(0, str(Path("/content/drive/MyDrive/Smart_Vision_AI")))
        sys.path.insert(0, str(PROJECT_ROOT))
    else:
        PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
        OUT_ROOT = PROJECT_ROOT
        sys.path.insert(0, str(PROJECT_ROOT))

    REPORTS = OUT_ROOT / "reports"
    FIGURES = REPORTS / "figures"
    MODELS = OUT_ROOT / "models"
    FIGURES.mkdir(parents=True, exist_ok=True)

    clf = json.loads((REPORTS / "classification_metrics.json").read_text(encoding="utf-8"))
    yolo = json.loads((REPORTS / "yolo_metrics.json").read_text(encoding="utf-8"))
    meta_path = PROJECT_ROOT / "smartvision_dataset" / "dataset_metadata.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    print("best CNN:", clf["best_classification_model"])
    print("YOLO mAP50 val:", yolo["val"]["map50"])


In [ ]:
if SKIP_COMPARISON:
    print('SKIP c_cmp_charts')
else:
    ## Comparison table + charts

    rows = []
    for name, m in clf["classification"].items():
        rows.append({
            "model": name,
            "task": "classification",
            "accuracy": m["accuracy"],
            "precision": m["precision_macro"],
            "recall": m["recall_macro"],
            "f1": m["f1_macro"],
            "top5": m["top5_accuracy"],
            "inference_ms": m["inference_ms"],
            "size_mb": m["model_size_mb"],
        })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    df.plot.bar(x="model", y=["accuracy", "f1", "top5"], ax=axes[0], rot=20)
    axes[0].set_ylim(0, 1); axes[0].set_title("Accuracy / F1 / Top-5")
    df.plot.bar(x="model", y="inference_ms", ax=axes[1], rot=20, legend=False, color="#2E86AB")
    axes[1].set_title("Inference ms / image")
    df.plot.bar(x="model", y="size_mb", ax=axes[2], rot=20, legend=False, color="#E94F37")
    axes[2].set_title("Model size (MB)")
    fig.tight_layout()
    fig.savefig(FIGURES / "model_comparison.png", dpi=140)
    plt.show()

    # Speed vs accuracy scatter
    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(df["inference_ms"], df["accuracy"], s=df["size_mb"] * 8)
    for _, r in df.iterrows():
        ax.annotate(r["model"], (r["inference_ms"], r["accuracy"]), textcoords="offset points", xytext=(6, 4))
    ax.set_xlabel("inference ms"); ax.set_ylabel("test accuracy")
    ax.set_title("Accuracy–speed tradeoff (bubble ~ size)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES / "accuracy_speed_tradeoff.png", dpi=140)
    plt.show()

    best = clf["best_classification_model"]
    print("Selected classification model (highest test accuracy):", best)
    print("YOLO is used for multi-object localization; CNN is optional verification on crops.")


In [ ]:
if SKIP_COMPARISON:
    print('SKIP c_cmp_pipe')
else:
    ## End-to-end pipeline on detection test images
    # YOLO NMS is internal; we filter confidence > 0.50 as the brief requires.

    from ultralytics import YOLO

    CLASS_NAMES = clf["class_names"]
    yolo_model = YOLO(str(MODELS / "yolov8_best.pt"))
    cnn_name = best
    cnn_path = {
        "VGG16": MODELS / "vgg16.keras",
        "ResNet50": MODELS / "resnet50.keras",
        "MobileNetV2": MODELS / "mobilenetv2.keras",
        "EfficientNetB0": MODELS / "efficientnetb0.keras",
    }[cnn_name]

    import tensorflow as tf
    cnn = tf.keras.models.load_model(cnn_path, compile=False)

    DET = PROJECT_ROOT / "smartvision_dataset" / "detection"
    if not (DET / "images" / "test").exists():
        DET = Path("/content/smartvision_dataset/detection")
    test_imgs = sorted((DET / "images" / "test").glob("*.jpg"))[:6]

    def ensure_rgb(im):
        return im.convert("RGB") if im.mode != "RGB" else im

    def classify_crop(pil_crop):
        arr = np.asarray(pil_crop.resize((224, 224)), dtype=np.float32)
        pred = cnn.predict(np.expand_dims(arr, 0), verbose=0)[0]
        i = int(pred.argmax())
        return CLASS_NAMES[i], float(pred[i])

    def run_pipeline(path, conf=0.50, verify_cnn=True):
        im = ensure_rgb(Image.open(path))
        res = yolo_model.predict(source=np.asarray(im), conf=conf, iou=0.45, verbose=False)[0]
        dets = []
        if res.boxes is not None:
            for b in res.boxes:
                x1, y1, x2, y2 = [float(v) for v in b.xyxy[0].tolist()]
                cls_id = int(b.cls[0]); score = float(b.conf[0])
                label = res.names.get(cls_id, CLASS_NAMES[cls_id])
                rec = {"xyxy": [x1, y1, x2, y2], "class": label, "confidence": score, "cnn_class": None, "agree": None}
                if verify_cnn:
                    crop = im.crop((max(0,int(x1)), max(0,int(y1)), int(x2), int(y2)))
                    if crop.size[0] > 2 and crop.size[1] > 2:
                        cc, cs = classify_crop(crop)
                        rec["cnn_class"] = cc
                        rec["cnn_confidence"] = cs
                        rec["agree"] = cc == label
                dets.append(rec)
        canvas = im.copy()
        dr = ImageDraw.Draw(canvas)
        for d in dets:
            x1, y1, x2, y2 = d["xyxy"]
            dr.rectangle([x1, y1, x2, y2], outline="red", width=3)
            tag = f"{d['class']} {d['confidence']:.2f}"
            if d.get("cnn_class"):
                tag += f" | CNN {d['cnn_class']} {d['cnn_confidence']:.2f}"
            dr.text((x1, max(0, y1 - 12)), tag, fill="red")
        return canvas, dets

    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    all_dets = []
    t0 = time.perf_counter()
    for ax, p in zip(axes.ravel(), test_imgs):
        canvas, dets = run_pipeline(p, conf=0.50, verify_cnn=True)
        all_dets.extend(dets)
        ax.imshow(canvas)
        ax.set_title(f"{p.name}  n={len(dets)}", fontsize=8)
        ax.axis("off")
    elapsed = time.perf_counter() - t0
    fig.suptitle(f"Pipeline YOLO+{cnn_name}  conf>0.5  ({elapsed/max(len(test_imgs),1):.2f}s / image)")
    fig.tight_layout()
    fig.savefig(FIGURES / "pipeline_samples.png", dpi=140)
    plt.show()
    if all_dets:
        agree = [d["agree"] for d in all_dets if d["agree"] is not None]
        print(f"CNN-YOLO agreement on {len(agree)} crops: {np.mean(agree):.3f}" if agree else "no crops")
    print("objects/image (this slice):", len(all_dets) / max(len(test_imgs), 1))


In [ ]:
if SKIP_COMPARISON:
    print('SKIP c_cmp_quant')
else:
    ## Quantization / export for cloud (Phase 4.3)
    # TFLite dynamic-range for the best CNN; ONNX for YOLO if export succeeds.

    best_path = {
        "VGG16": MODELS / "vgg16.keras",
        "ResNet50": MODELS / "resnet50.keras",
        "MobileNetV2": MODELS / "mobilenetv2.keras",
        "EfficientNetB0": MODELS / "efficientnetb0.keras",
    }[clf["best_classification_model"]]

    converter = tf.lite.TFLiteConverter.from_keras_model(tf.keras.models.load_model(best_path, compile=False))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    tflite_path = MODELS / "best_cnn_dynamic.tflite"
    tflite_path.write_bytes(tflite_bytes)
    print("TFLite", tflite_path, "MB", len(tflite_bytes)/1e6)

    try:
        YOLO = __import__("ultralytics").YOLO
        y = YOLO(str(MODELS / "yolov8_best.pt"))
        y.export(format="onnx", imgsz=640, simplify=True)
        print("ONNX export attempted in the YOLO run directory / models folder")
    except Exception as e:
        print("ONNX export skipped:", e)

    quant = {
        "tflite": str(tflite_path),
        "tflite_mb": len(tflite_bytes) / 1e6,
        "original_mb": clf["classification"][clf["best_classification_model"]]["model_size_mb"],
    }
    print(quant)


In [ ]:
if SKIP_COMPARISON:
    print('SKIP c_cmp_metrics')
else:
    ## Final metrics.json consumed by the Streamlit Performance page

    metrics = {
        "class_names": clf["class_names"],
        "dataset": meta.get("classification", {}),
        "dataset_decisions": meta.get("decisions", {}),
        "classification": clf["classification"],
        "best_classification_model": clf["best_classification_model"],
        "yolo": {
            "model": yolo.get("model", "YOLOv8s"),
            "map50": yolo["val"]["map50"],
            "map50_95": yolo["val"]["map50_95"],
            "precision": yolo["val"]["precision"],
            "recall": yolo["val"]["recall"],
            "fps": yolo.get("fps"),
            "test": yolo.get("test", {}),
            "per_class_ap50": yolo.get("per_class_ap50", {}),
            "meets_map50_floor": yolo.get("meets_map50_floor"),
        },
        "quantization": quant,
    }
    (REPORTS / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print("Wrote", REPORTS / "metrics.json")
    print("Download Drive/SmartVision_artifacts/{models,reports} into the GitHub repo.")


## Done

Artifacts are on Drive: `MyDrive/SmartVision_artifacts/{models,reports}`.

Paste into chat:
1. The classification accuracy table (Phase 2 persist cell)
2. `VAL extracted {'map50': ...}` and `RUBRIC: val mAP@0.5 > 0.75 ?`
3. `best CNN:` and `YOLO mAP50 val:` from Phase 4

Keep notebooks `02_`, `03_`, `04_` in the repo for the rubric; this file is only so you do not change runtimes.
